# Transformer-JEPA v1 hard-disjoint VICReg — 12x12 Colab Training

This run uses `configs/jepa_v1_vicreg_b12_hard_disjoint.yml`. The launcher loads it automatically.

Run the cells in order. Run the full-training cell only after the smoke gate passes.

## 1. Mount Drive and install the project

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, subprocess, sys

PROJECT_ROOT = Path('/content/drive/Othercomputers/MyLaptop/Master_Thesis_Code')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/Master_Thesis_Artifacts')
DATA_DIR = ARTIFACT_ROOT / 'data' / 'othello_12x12' / 'corpus_v1'

assert (PROJECT_ROOT / 'scripts' / 'launch_12x12_training.py').is_file(), PROJECT_ROOT
assert (DATA_DIR / 'manifest.json').is_file(), DATA_DIR
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT_ROOT)], check=True)
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_DIR    :', DATA_DIR)

def run_streaming(command):
    # Forward child output as it arrives, including '\r'-refreshed progress
    # updates, so the trainer's in-place chunk progress bar renders live.
    # Line-based iteration would buffer the bar until the next newline.
    import codecs
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    print('COMMAND:', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(
        list(map(str, command)),
        cwd=PROJECT_ROOT,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    assert process.stdout is not None
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    while True:
        data = process.stdout.read1(8192)
        if not data:
            break
        sys.stdout.write(decoder.decode(data))
        sys.stdout.flush()
    tail = decoder.decode(b'', True)
    if tail:
        sys.stdout.write(tail)
        sys.stdout.flush()
    returncode = process.wait()
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, command)
    return returncode


Mounted at /content/drive
PROJECT_ROOT: /content/drive/Othercomputers/MyLaptop/Master_Thesis_Code
DATA_DIR    : /content/drive/MyDrive/Master_Thesis_Artifacts/data/othello_12x12/corpus_v1


## 3. Validate the exact corpus and split

In [2]:
import json
manifest = json.loads((DATA_DIR / 'manifest.json').read_text())
expected = {
    'complete': True, 'board_size': 12, 'generated_games': 23_800_000,
    'training_games': 20_000_000, 'heldout_games': 3_800_000,
    'model_block_size': 139,
}
for key, value in expected.items():
    assert manifest.get(key) == value, (key, manifest.get(key), value)
assert len(manifest['shards']) == 238
print('12x12 corpus gate: PASS')


12x12 corpus gate: PASS


## 4. Smoke gate — one training shard

This writes to a separate `_smoke` run and cannot overwrite the full run.

In [3]:
print('STARTING smoke: transformer_jepa', flush=True)
run_streaming([sys.executable, '-u', 'scripts/launch_12x12_training.py', 'transformer_jepa', '--smoke'])
print('SMOKE FINISHED: transformer_jepa', flush=True)


STARTING smoke: transformer_jepa
COMMAND: /usr/bin/python3 -u scripts/launch_12x12_training.py transformer_jepa --smoke
[12x12 launcher] START experiment=transformer_jepa smoke=True
[12x12 launcher] validating corpus manifest...
[12x12 launcher] corpus validation: PASS
[12x12 launcher] command: /usr/bin/python3 -u -m othello_research.training.train_jepa --config /content/drive/Othercomputers/MyLaptop/Master_Thesis_Code/configs/jepa_v1_vicreg_b12_hard_disjoint.yml --data_dir /content/drive/MyDrive/Master_Thesis_Artifacts/data/othello_12x12/corpus_v1 --li_split_train_chunks 200 --out_dir /content/runs/Transformer-JEPA/transformer_jepa_v1_hd_vicreg_b12_smoke --drive_sync_dir /content/drive/MyDrive/Master_Thesis_Artifacts/runs/Transformer-JEPA/transformer_jepa_v1_hd_vicreg_b12_smoke --drive_sync_every_chunks 5 --board_size 12 --batch_size 256 --precision bf16 --passes_over_data 1 --checkpoint_every_chunks 0 --milestone_games 1000000,5000000,10000000,15000000,20000000 --seed 42 --max_chunks

## 5. Full 20M-game training

Run this cell only after smoke loss is finite and there was no OOM.

In [4]:
print('STARTING full training: transformer_jepa', flush=True)
run_streaming([sys.executable, '-u', 'scripts/launch_12x12_training.py', 'transformer_jepa'])
print('FULL TRAINING FINISHED: transformer_jepa', flush=True)


STARTING full training: transformer_jepa
COMMAND: /usr/bin/python3 -u scripts/launch_12x12_training.py transformer_jepa
[12x12 launcher] START experiment=transformer_jepa smoke=False
[12x12 launcher] validating corpus manifest...
[12x12 launcher] corpus validation: PASS
[12x12 launcher] command: /usr/bin/python3 -u -m othello_research.training.train_jepa --config /content/drive/Othercomputers/MyLaptop/Master_Thesis_Code/configs/jepa_v1_vicreg_b12_hard_disjoint.yml --data_dir /content/drive/MyDrive/Master_Thesis_Artifacts/data/othello_12x12/corpus_v1 --li_split_train_chunks 200 --out_dir /content/runs/Transformer-JEPA/transformer_jepa_v1_hd_vicreg_b12 --drive_sync_dir /content/drive/MyDrive/Master_Thesis_Artifacts/runs/Transformer-JEPA/transformer_jepa_v1_hd_vicreg_b12 --drive_sync_every_chunks 5 --board_size 12 --batch_size 256 --precision bf16 --passes_over_data 1 --checkpoint_every_chunks 0 --milestone_games 1000000,5000000,10000000,15000000,20000000 --seed 42
[12x12 launcher] traine

## 6. Resume after a disconnected runtime (optional)

Set `RESUME_CHECKPOINT` to a real synced checkpoint, then run the cell.

In [5]:
RESUME_CHECKPOINT = None  # e.g. '/content/drive/MyDrive/.../latest.pt'
if RESUME_CHECKPOINT:
    run_streaming([sys.executable, '-u', 'scripts/launch_12x12_training.py', 'transformer_jepa', '--resume-from', RESUME_CHECKPOINT])
else:
    print('Resume skipped: RESUME_CHECKPOINT is None')


Resume skipped: RESUME_CHECKPOINT is None
